# Project 1: Group 42

Text classification on **SST-2** (binary sentiment). Task: `project.ipynb`. Code reused from `tutorial.ipynb` and `part1/*.py`.

Plan (section N = backup slide N):
1. Data and metric
2. Tokenizer
3. Representations: indicator, bag-of-words, + our choice
4. Architectures: log-linear, MLP, + our choice
5. Evaluation: cross-validation setup and metric
6. Analysis: Bayesian comparison with a ROPE, error analysis

> Use of AI tools: notebook scaffold (structure, setup/experiment/analysis helper code, TODO prompts) generated with Claude; implementations, experiments and conclusions are ours.

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
from pathlib import Path
from collections import Counter, OrderedDict

import json
import numpy as np
import matplotlib.pyplot as plt
import torch

# reproducibility: the CV seeds only fix the fold split; these fix model training
# (loglin.fit uses sklearn's saga solver without random_state, so it falls back on np.random)
np.random.seed(0)
torch.manual_seed(0)

root = Path.cwd().parent
sys.path.insert(0, str(root))
import part1

results_folder = Path("./results/p1_group42")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data and metric

SST-2 ([course dataset list](https://nlp-illc.github.io/nlp1/2026/datasets/)): `stanfordnlp/sst2`.
The test split has no labels (all -1), so we cross-validate on train and keep validation as our held-out test set.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")
print(dataset)

outcome_space = part1.clslib.NamedClasses(['negative', 'positive'])
print(outcome_space)

train_pairs = list(zip(dataset["train"]["sentence"], dataset["train"]["label"]))
heldout_pairs = list(zip(dataset["validation"]["sentence"], dataset["validation"]["label"]))   # held-out, use only at the very end

In [ ]:
# TODO (slide 1): dataset statistics
print("Train Size: ", len(train_pairs))
print("Validation Size: ", len(heldout_pairs),"\n")

def label_stats(pairs, name):
    print(f"{name} labels:")
    labels = [label for text, label in pairs]
    labels = np.array(labels)
    print(" Positive: ", np.sum(labels))
    print(" Negative: ", len(pairs)-np.sum(labels))
    print(" Positive fraction: ", np.mean(labels))
    print(" Negative fraction: ", 1-np.mean(labels),"\n")

#Train labels:
label_stats(train_pairs, "Train")

#Validation labels:
label_stats(heldout_pairs, "Validation")

## 2. Tokenizer

Questions to answer (slide 2):
- train our own BPE or reuse the tutorial's (trained on AG News)?
- vocab_size / min_frequency: why these values?
- which corpus do we train it on? Is the train split enough?
- the tokenizer must not see the held-out set
- the tokenizer is trained before the CV split: does it matter that it sees the validation folds?

In [ ]:
vocab_size = 4000      # TODO: choose and motivate
min_frequency = 2
tokenizer_corpus = [text for text, _ in train_pairs]   # TODO: decide on the corpus

tokenizer = part1.experiment.train_tokenizer(tokenizer_corpus, vocab_size, min_frequency)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}_mf{min_frequency}.json")

print(tokenizer.encode(train_pairs[0][0]).tokens)

In [ ]:
# TODO (slide 2): tokenizer sanity checks


## 5a. Cross-validation splits

(Section 5 of the plan; created here because sections 3 and 4 need the splits.)

In [ ]:
my_seeds = [0]
num_folds = 5

my_cross_validation_sets = [
    part1.experiment.cross_validation_split(pairs=train_pairs, k=num_folds, seed=seed)
    for seed in my_seeds
]

## 3. Representations

Note: `part1.clslib.TokenIndicator.__len__` uses the global variable `tokenizer` (a bug in the course code).
It works as long as our tokenizer is called `tokenizer`; the copy below fixes it.

In [ ]:
from tokenizers import Tokenizer

class TokenIndicator(part1.clslib.Encoder):
    """h_v(x) = 1 if BPE token v occurs in x"""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self):
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        vector[list(set(self.tokenizer.encode(x).ids))] = 1
        return vector

In [ ]:
class BagOfWords(part1.clslib.Encoder):
    """h_v(x) = number of times BPE token v occurs in x"""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self):
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> np.ndarray:
        # TODO: like TokenIndicator, but count occurrences (hint: np.add.at or collections.Counter)
        raise NotImplementedError

In [ ]:
class OurEncoder(part1.clslib.Encoder):
    """TODO (slide 3): third representation of our choice. Motivate why it should help for *our* task.
    If it needs statistics from data, think about which data you compute them on."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self):
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

In [ ]:
encoders = {
    'indicator': TokenIndicator(tokenizer),
    'bow': BagOfWords(tokenizer),
    # 'ours': OurEncoder(tokenizer),
}
# sanity check: dimensions and an example
for name, enc in encoders.items():
    h = enc(train_pairs[0][0])
    print(name, len(enc), h.sum())

## 4. Architectures and experiments

`part1.experiment.cross_validation_experiment` supports `model_option='logistic'` (log-linear) and `'mlp'` (1 hidden layer).
For the third architecture, copy `smlp.py` to e.g. `smlp_ours.py`, change `SparseMLP` (more layers, another activation, dropout, ...)
and add a branch to a copy of `cross_validation_experiment`.

In [ ]:
my_classifiers = OrderedDict()
scores_folder = Path("./p1_scores")   # tracked in git: fold accuracies only, no models
scores_folder.mkdir(exist_ok=True)

def run(name, encoder, model_option, **kwargs):
    """Cross-validate one system for every seed; results go to my_classifiers[name] and p1_scores/{name}.json"""
    results = []
    for run_id, seed in enumerate(my_seeds):
        results.append(
            part1.experiment.cross_validation_experiment(
                data_splits=my_cross_validation_sets[run_id],
                outcome_space=outcome_space,
                encoder=encoder,
                model_option=model_option,
                save_to=f"{results_folder}/{name}/seed-{seed}",
                **kwargs,
            )
        )
    my_classifiers[name] = results   # only stored if every run finished
    with open(scores_folder / f"{name}.json", "w") as f:
        json.dump([{'accuracies': {int(k): float(v) for k, v in r['accuracies'].items()}} for r in results], f)

def load_scores():
    """Load every teammate's pushed fold accuracies (systems not run in this kernel)."""
    for path in sorted(scores_folder.glob("*.json")):
        if path.stem not in my_classifiers:
            with open(path) as f:
                my_classifiers[path.stem] = json.load(f)

In [ ]:
%%time
for enc_name, enc in encoders.items():
    run(f"loglin_{enc_name}", enc, 'logistic', loglin_args={'max_iter': 100})

In [ ]:
%%time
mlp_args = {'hidden_dim': 128, 'dropout': 0.1, 'batch_size': 64, 'learning_rate': 1e-3,
            'weight_decay': 1e-4, 'num_epochs': 10}   # TODO: motivate the choices
for enc_name, enc in encoders.items():
    run(f"mlp_{enc_name}", enc, 'mlp', smlp_args=mlp_args)

In [ ]:
# TODO (slide 4): third architecture on all three representations

## 5b + 6. Results and analysis

Slide 5: is the evaluation adequate (metric, CV setup)? Slide 6: which system is best, and how sure are we?

In [ ]:
load_scores()
df = part1.analysis.make_data_frame(my_classifiers)
display(df.groupby('model').accuracy.agg(['mean', 'std']).sort_values('mean', ascending=False))
part1.analysis.plot_box_and_points(df)
# TODO (slide 5): compare against the majority-class baseline

In [ ]:
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

ROPE = None   # TODO (slide 6): choose and motivate (what difference in accuracy matters in practice for us?)

def compare(a, b, rope=None, plot=False):
    rope = ROPE if rope is None else rope
    assert rope is not None, "set ROPE first"
    post = part1.analysis.BenavoliCorrelatedT(
        get_all_measurements_for(my_classifiers, a).flatten(),
        get_all_measurements_for(my_classifiers, b).flatten(),
        runs=len(my_seeds),   # the tutorial omits this; with several seeds it must be set
        names=[a, b],
    )
    p_a, p_tie, p_b = post.probs(rope)
    print(f"A={a} B={b}: P(A wins)={p_a:.3f} P(tie)={p_tie:.3f} P(B wins)={p_b:.3f} mean diff={post.mean_diff:+.4f}")
    if plot:
        post.plot(rope)

compare('loglin_indicator', 'loglin_bow', plot=True)
# TODO: which comparisons answer our questions? (representation effect per architecture, architecture effect per representation)

In [ ]:
# TODO (slide 6): error analysis

In [ ]:
# TODO (final): retrain the chosen system on all of train_pairs and report accuracy on heldout_pairs (only once!)